# Fine Tuning The Model

In [1]:
%%writefile train.py
"""
LoRA fine-tuning of BioMistral-7B on lavita/medical-qa-datasets (all-processed)
Target: Kaggle 2x T4 (16GB each) — TRUE DUAL-GPU via DDP
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
import threading
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "BioMistral/BioMistral-7B"
DATASET_NAME = "lavita/medical-qa-datasets"
DATASET_CONFIG = "all-processed"
OUTPUT_DIR = "./finetuned-model"
MAX_LENGTH = 1024

local_rank = int(os.environ.get("LOCAL_RANK", 0))
is_main_process = local_rank == 0

raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)

if is_main_process:
    print("GPUs visible:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": local_rank},
)
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
if is_main_process:
    model.print_trainable_parameters()

def format_and_tokenize(example):
    prompt = f"{example['instruction']}\n\n{example['input']}\n\n### Response:\n"
    response = example["output"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH, add_special_tokens=True)["input_ids"]
    response_ids = tokenizer(response, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + response_ids
    input_ids = input_ids[:MAX_LENGTH]

    labels = [-100] * len(prompt_ids) + response_ids
    labels = labels[:MAX_LENGTH]

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

tokenized = raw_dataset.map(
    format_and_tokenize,
    remove_columns=raw_dataset["train"].column_names,
)

if "validation" in tokenized:
    train_ds, eval_ds = tokenized["train"], tokenized["validation"]
else:
    split = tokenized["train"].train_test_split(test_size=0.05, seed=42)
    train_ds, eval_ds = split["train"], split["test"]

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

class LiveProgressMonitor(TrainerCallback):
    def __init__(self, interval_sec=1):
        self.interval_sec = interval_sec
        self.total_steps = None
        self.start_time = None
        self.current_step = 0
        self.stop_flag = False
        self.thread = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.total_steps = state.max_steps
        self.start_time = time.time()
        self.stop_flag = False
        if is_main_process:
            self.thread = threading.Thread(target=self._monitor_loop, daemon=True)
            self.thread.start()

    def on_step_end(self, args, state, control, **kwargs):
        self.current_step = state.global_step

    def on_train_end(self, args, state, control, **kwargs):
        self.stop_flag = True
        if self.thread is not None:
            self.thread.join(timeout=2)

    def _monitor_loop(self):
        while not self.stop_flag:
            elapsed = time.time() - self.start_time
            pct = (self.current_step / self.total_steps * 100) if self.total_steps else 0
            eta = (elapsed / self.current_step * (self.total_steps - self.current_step)
                   if self.current_step > 0 else 0)

            gpu_status = " | ".join(
                f"GPU{i}: {torch.cuda.memory_allocated(i) / 1024**3:.1f}GB"
                for i in range(torch.cuda.device_count())
            )

            print(
                f"\r[{pct:5.1f}%] step {self.current_step}/{self.total_steps} "
                f"| elapsed {elapsed:6.0f}s | ETA {eta:6.0f}s | {gpu_status}",
                end="", flush=True,
            )
            time.sleep(self.interval_sec)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
    ddp_find_unused_parameters=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    callbacks=[LiveProgressMonitor(interval_sec=1)],
)

trainer.train()

if is_main_process:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"\nLoRA adapter saved to {OUTPUT_DIR}")

Writing train.py


In [ ]:
!torchrun --nproc_per_node=2 train.py

W0921 09:53:26.501000 122 torch/distributed/run.py:852] 
W0921 09:53:26.501000 122 torch/distributed/run.py:852] *****************************************
W0921 09:53:26.501000 122 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0921 09:53:26.501000 122 torch/distributed/run.py:852] *****************************************
README.md: 16.4kB [00:00, 6.32MB/s]
all-processed/train-00000-of-00001-a77e2(…):   0%|   | 0.00/155M [00:02<?, ?B/s]

In [ ]:
import torch
print(torch.cuda.device_count())          # should print 2
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_name(1))